# 7.10 — ResNet & Residual Learning

ResNet made very deep vision networks trainable by changing the job of a block: instead of rebuilding an entire representation from scratch, the learned layers predict a correction that is added to an identity shortcut. In this lesson, we build residual addition, shape matching, projection shortcuts, and gradient flow from scratch with NumPy so the architecture feels like arithmetic rather than magic.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build residual learning one idea at a time. Run each cell in order and read the printed intermediate values — every shortcut, correction, shape check, and gradient term is made explicit. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized arithmetic, and small linear maps for residual blocks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for tiny learned-weight demos.

### 1. A plain block must rebuild the whole target

A plain deep block tries to learn the full mapping $H(x)$ directly. If the best thing a layer can do is mostly preserve the input, the plain block still has to discover an identity-like transformation through its weights. That sounds easy in words, but many nonlinear layers stacked together must all avoid damaging useful information while also transforming it.

In [ ]:
x_w = np.array([1.0, 2.0, 3.0])  # an input representation with three feature coordinates.
H_target_w = np.array([1.1, 1.8, 3.3])  # the desired output after a useful small change.
plain_needed_w = H_target_w.copy()  # a plain block must emit the whole target vector.

print("input x:", x_w)  # inspect the signal entering the block.
print("target H(x):", H_target_w)  # inspect the full signal the plain block must create.

▶ What you'll see: the target is very close to the input, but the plain block still owns every coordinate of the output.

In [ ]:
plt.figure(figsize=(4.4, 3))  # compare input and target coordinate by coordinate.
plt.plot(x_w, marker="o", label="input x")  # show the identity path value.
plt.plot(H_target_w, marker="s", label="target H(x)")  # show the desired block output.
plt.title("1: target is mostly the input")  # title the comparison.
plt.xlabel("feature coordinate")  # label coordinates.
plt.ylabel("value")  # label feature scale.
plt.legend()  # show line meanings.
plt.show()  # display the plot.

▶ What you'll see: the two lines nearly overlap, so most of the desired mapping is preservation.

*Why it's done this way:* Writing the target as $H(x)$ highlights the optimization burden in a plain stack: every layer must learn both “keep what matters” and “change what needs changing.” When depth grows, repeatedly relearning preservation is fragile because small deviations from identity compound across many layers.

### 2. A residual block learns only the correction

A residual block rewrites the target as $H(x)=x+F(x)$. The learned branch $F$ no longer has to recreate the whole signal; it only supplies the difference $H(x)-x$. If the input is already good, the correction can be small.

In [ ]:
residual_w = H_target_w - x_w  # the correction F(x) needed to reach the target.
y_w = x_w + residual_w  # residual block output y = x + F(x).

print("correction F(x):", residual_w)  # inspect the small learned change.
print("x + F(x):", y_w)  # inspect the residual output.

assert np.allclose(y_w, H_target_w)  # verify residual addition exactly recovers the target.

▶ What you'll see: the correction `[0.1, -0.2, 0.3]` is much smaller than the full target `[1.1, 1.8, 3.3]`.

In [ ]:
plt.figure(figsize=(4.4, 3))  # visualize the small residual branch.
plt.bar(["coord0", "coord1", "coord2"], residual_w, color="teal")  # plot F(x) by coordinate.
plt.axhline(0, color="black", linewidth=0.8)  # separate positive and negative corrections.
plt.title("2: residual branch learns the correction")  # title the correction plot.
plt.ylabel("F(x)")  # label residual value.
plt.show()  # display the bar chart.

▶ What you'll see: the learned branch makes only local coordinate adjustments around the identity path.

*Why it's done this way:* Subtracting $x$ from the desired output isolates the mathematical residual. The architecture then gives the model a cheap default answer — pass $x$ through — and asks the learned branch to spend capacity only where the representation needs to move.

### 3. Identity is easy when the correction is zero

The special case $F(x)=0$ is the core ResNet trick. A residual block can become an identity map without learning a full identity matrix; it only needs the residual branch to produce values near zero. That makes extra depth less harmful when extra transformations are not useful.

In [ ]:
zero_residual_w = np.zeros_like(x_w)  # a residual branch that chooses no correction.
identity_out_w = x_w + zero_residual_w  # residual addition with F(x)=0.

print("F(x)=", zero_residual_w)  # inspect the no-op correction.
print("output:", identity_out_w)  # inspect that the block returns x unchanged.

assert np.allclose(identity_out_w, x_w)  # verify exact identity behavior.

▶ What you'll see: adding a zero correction gives `[1, 2, 3]` back exactly.

In [ ]:
scales_w = np.linspace(0, 1, 6)  # gradually turn the correction on.
outputs_w = np.array([x_w + s_w * residual_w for s_w in scales_w])  # interpolate identity to target.

print("first output:", outputs_w[0], "last output:", outputs_w[-1])  # inspect endpoints.

plt.figure(figsize=(4.5, 3))  # show how residual strength moves the output.
plt.plot(scales_w, outputs_w[:, 0], marker="o", label="coord0")  # coordinate 0 path.
plt.plot(scales_w, outputs_w[:, 1], marker="o", label="coord1")  # coordinate 1 path.
plt.plot(scales_w, outputs_w[:, 2], marker="o", label="coord2")  # coordinate 2 path.
plt.title("3: zero residual starts at identity")  # title interpolation.
plt.xlabel("residual scale")  # label scale.
plt.ylabel("output coordinate")  # label output value.
plt.legend()  # show coordinates.
plt.show()  # display the plot.

▶ What you'll see: scale 0 is exactly the input, and scale 1 reaches the target correction.

*Why it's done this way:* Initializing or learning a residual branch near zero makes a deep block behave like a safe no-op. Mathematically, the identity term is already present in $x+F(x)$, so the optimizer does not need to synthesize identity through many weights before it can preserve information.

### 4. Addition forces exact shape matching

Residual addition is elementwise, so the shortcut tensor and residual branch output must have the same shape. Addition is not concatenation: it keeps the feature count fixed and requires both paths to speak in the same coordinate system.

In [ ]:
shortcut_w = np.ones((2, 3, 4))  # height=2, width=3, channels=4.
branch_good_w = 0.1 * np.ones((2, 3, 4))  # matching residual branch shape.
branch_bad_w = 0.1 * np.ones((2, 3, 5))  # wrong channel count for residual addition.

print("shortcut shape:", shortcut_w.shape)  # inspect shortcut dimensions.
print("matching branch shape:", branch_good_w.shape)  # inspect valid branch dimensions.
print("mismatched branch shape:", branch_bad_w.shape)  # inspect invalid branch dimensions.

▶ What you'll see: the first two tensors match exactly; the third has 5 channels instead of 4.

In [ ]:
added_good_w = shortcut_w + branch_good_w  # valid elementwise residual addition.
can_add_bad_w = shortcut_w.shape == branch_bad_w.shape  # explicit shape check before adding.

print("valid output shape:", added_good_w.shape)  # inspect unchanged shape after addition.
print("can add mismatched tensors?", can_add_bad_w)  # inspect why the bad branch is invalid.

assert added_good_w.shape == shortcut_w.shape  # addition preserves shape when paths match.
assert can_add_bad_w is False  # mismatched channels cannot be added safely.

▶ What you'll see: the valid residual sum stays `(2, 3, 4)`, while the mismatched case is rejected.

*Why it's done this way:* Elementwise addition combines coordinate $c$ from the shortcut with coordinate $c$ from the residual branch. If the channel counts or spatial locations differ, there is no unambiguous coordinate-by-coordinate correction, so the architecture must align shapes before summing.

### 5. Projection shortcuts fix channel mismatch

When a block changes the number of channels, the shortcut cannot be raw identity anymore. A $1\times1$ projection learns a linear map at each spatial location, changing the shortcut from $C_{in}$ channels to $C_{out}$ channels so it can be added to the residual branch.

In [ ]:
x_proj_w = np.array([[[1.0, 2.0, 3.0], [0.0, 1.0, 0.5]]])  # one row, two positions, three channels.
W_proj_w = np.array([[1.0, 0.0, 0.5, -0.5],  # 3 input channels -> 4 output channels.
                     [0.0, 1.0, 0.5, 0.5],
                     [0.0, 0.0, 1.0, 1.0]])
shortcut_proj_w = x_proj_w @ W_proj_w  # apply the same 1x1 linear map at each spatial position.

print("input shape:", x_proj_w.shape)  # inspect H x W x C_in.
print("projected shortcut shape:", shortcut_proj_w.shape)  # inspect H x W x C_out.

▶ What you'll see: the shortcut changes from 3 channels to 4 channels while keeping spatial positions.

In [ ]:
weights_64_128_w = 1 * 1 * 64 * 128  # parameter count for a 1x1 projection from 64 to 128 channels.
residual_branch_w = 0.05 * np.ones_like(shortcut_proj_w)  # branch now matches projected shortcut.
y_proj_w = shortcut_proj_w + residual_branch_w  # valid addition after projection.

print("1x1 projection weights 64→128:", weights_64_128_w)  # inspect concrete cost.
print("output shape after add:", y_proj_w.shape)  # inspect matched residual output.

assert weights_64_128_w == 8192  # verify the lesson's concrete projection count.
assert y_proj_w.shape == residual_branch_w.shape  # projection made addition legal.

▶ What you'll see: a 64→128 shortcut projection uses 8192 weights, and projected tensors add cleanly.

*Why it's done this way:* A projection is more than a shape hack. It learns a new coordinate system for the shortcut so that adding it to the residual branch is meaningful; the price is extra parameters and the loss of a pure identity shortcut in that block.

### 6. The shortcut gives gradients a direct additive route

For a scalar residual block $y=x+F(x)$, the derivative is $dy/dx=1+dF/dx$. During backpropagation, $dL/dx=(dL/dy)(1+dF/dx)$. The `1` is the shortcut's direct route; a plain branch would only multiply by $dF/dx$.

In [ ]:
dL_dy_w = 2.0  # upstream gradient from later layers.
dF_dx_w = 0.1  # local derivative through the learned residual branch.
dL_dx_res_w = dL_dy_w * (1.0 + dF_dx_w)  # residual block gradient.
dL_dx_plain_w = dL_dy_w * dF_dx_w  # branch-only gradient without shortcut identity term.

print("residual dL/dx:", round(dL_dx_res_w, 3))  # inspect shortcut-aided gradient.
print("plain branch-only dL/dx:", round(dL_dx_plain_w, 3))  # inspect much smaller branch-only flow.

assert round(dL_dx_res_w, 3) == 2.2  # verify concrete derivative from the lesson.
assert round(dL_dx_plain_w, 3) == 0.2  # verify branch-only contribution.

▶ What you'll see: the residual path sends back 2.2, while the branch alone would send only 0.2.

In [ ]:
branch_derivs_w = np.array([0.8, 0.5, 0.2, 0.1])  # four small derivatives in a deep plain stack.
plain_product_w = float(np.prod(branch_derivs_w))  # plain gradient multiplier through all branches.
residual_product_w = float(np.prod(1.0 + branch_derivs_w))  # residual-style multipliers include identity terms.

print("plain multiplier:", round(plain_product_w, 4))  # inspect multiplicative shrinkage.
print("residual-style multiplier:", round(residual_product_w, 4))  # inspect additive identity support.

plt.figure(figsize=(4.5, 3))  # compare gradient multipliers.
plt.bar(["plain product", "with identity terms"], [plain_product_w, residual_product_w], color=["crimson", "seagreen"])  # plot both routes.
plt.title("6: identity terms support gradient flow")  # title plot.
plt.ylabel("gradient multiplier")  # label multiplier scale.
plt.show()  # display comparison.

▶ What you'll see: multiplying small derivatives shrinks quickly, while identity terms keep a much stronger route.

*Why it's done this way:* Residual connections do not guarantee perfect optimization, but the derivative algebra changes the backward signal from “only the learned branch product” to “identity plus learned branch.” That additive term makes it easier for gradients to cross many blocks without disappearing solely because every branch derivative is small.

### 7. Downsampling residual blocks must align both paths

Some ResNet blocks reduce spatial size and increase channels, for example from $56\times56\times64$ to $28\times28\times128$. Then both the residual branch and shortcut must downsample and project so their tensors meet at the same shape before addition.

In [ ]:
H_in_w, W_in_w, C_in_w = 56, 56, 64  # incoming feature map shape.
stride_w, C_out_w = 2, 128  # downsampling stride and output channels.
H_out_w = H_in_w // stride_w  # stride-2 halves the spatial size in this clean example.
W_out_w = W_in_w // stride_w  # stride-2 halves width too.

print("input shape:", (H_in_w, W_in_w, C_in_w))  # inspect starting shape.
print("target branch shape:", (H_out_w, W_out_w, C_out_w))  # inspect output shape.

assert (H_out_w, W_out_w, C_out_w) == (28, 28, 128)  # verify the lesson shape.

▶ What you'll see: the block target is `(28, 28, 128)`, not compatible with a raw `(56, 56, 64)` shortcut.

In [ ]:
shortcut_raw_shape_w = (H_in_w, W_in_w, C_in_w)  # raw identity shortcut shape.
shortcut_projected_shape_w = (H_out_w, W_out_w, C_out_w)  # strided projection shortcut shape.
needs_projection_w = shortcut_raw_shape_w != shortcut_projected_shape_w  # check whether identity is legal.

print("raw shortcut shape:", shortcut_raw_shape_w)  # inspect no-projection shortcut.
print("projected shortcut shape:", shortcut_projected_shape_w)  # inspect aligned shortcut.
print("projection required?", needs_projection_w)  # inspect the design decision.

assert needs_projection_w is True  # downsampling and channel change require projection.

▶ What you'll see: the shortcut must change both spatial size and channel count to join the residual branch.

*Why it's done this way:* Residual addition is simple only after the architecture has done careful bookkeeping. The learned branch can use stride 2 to reduce the grid, but the shortcut must follow the same spatial stride and a $1\times1$ channel projection so the final sum is coordinate-aligned.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · A plain block must emit the whole target

If the desired output is close to the input, a plain block still has to produce every target coordinate itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)  # seeded for reproducibility.
t1_x = np.array([1.0, 3.0, 2.0])  # input representation.
t1_target = np.array([1.2, 2.5, 2.1])  # desired H(x).

print("input x:", t1_x.tolist())  # -> [1.0, 3.0, 2.0]
print("target H(x):", t1_target.tolist())  # -> [1.2, 2.5, 2.1]

t1_plain_needed = t1_target.copy()  # -> [1.2, 2.5, 2.1]

print("plain block must output:", t1_plain_needed.tolist())  # -> [1.2, 2.5, 2.1]

t1_plain_error_if_identity = t1_target - t1_x  # -> [0.19999999999999996, -0.5, 0.10000000000000009]

print("error if plain block only copied x:", np.round(t1_plain_error_if_identity, 3).tolist())  # -> [0.2, -0.5, 0.1]

assert np.allclose(t1_plain_needed, t1_target)

plt.figure(figsize=(4.4, 2.8))
plt.plot(t1_x, marker="o", label="input x")
plt.plot(t1_target, marker="s", label="target H(x)")
plt.title("Toy 1 · plain target is the full output")
plt.xlabel("coordinate")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: the target line nearly overlaps the input, but the plain block must still output all target values.

### ✍️ Toy 2 · Residual learning isolates the correction

A residual block can keep the shortcut and ask the learned branch to emit only `H(x) - x`.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)  # seeded for reproducibility.
t2_x = np.array([1.0, 3.0, 2.0])  # shortcut input.
t2_target = np.array([1.2, 2.5, 2.1])  # desired H(x).

print("input x:", t2_x.tolist())  # -> [1.0, 3.0, 2.0]
print("target H(x):", t2_target.tolist())  # -> [1.2, 2.5, 2.1]

t2_residual = t2_target - t2_x  # -> [0.19999999999999996, -0.5, 0.10000000000000009]

print("correction F(x):", np.round(t2_residual, 3).tolist())  # -> [0.2, -0.5, 0.1]

t2_y = t2_x + t2_residual  # -> [1.2, 2.5, 2.1]

print("x + F(x):", t2_y.tolist())  # -> [1.2, 2.5, 2.1]

assert np.allclose(t2_y, t2_target)

plt.figure(figsize=(4.4, 2.8))
plt.bar(["coord0", "coord1", "coord2"], t2_residual, color=["#54a24b", "#e45756", "#54a24b"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Toy 2 · residual branch learns the delta")
plt.ylabel("F(x)")
plt.show()

▶ What you'll see: the learned correction is much smaller than the whole target vector.

### ✍️ Toy 3 · Zero correction gives an identity block

When `F(x)=0`, residual addition returns the input exactly, so extra depth can start as a safe no-op.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)  # seeded for reproducibility.
t3_x = np.array([2.0, -1.0, 0.5])  # input representation.
t3_zero = np.zeros_like(t3_x)  # -> [0.0, 0.0, 0.0]

print("input x:", t3_x.tolist())  # -> [2.0, -1.0, 0.5]
print("zero correction:", t3_zero.tolist())  # -> [0.0, 0.0, 0.0]

t3_identity = t3_x + t3_zero  # -> [2.0, -1.0, 0.5]

print("identity output:", t3_identity.tolist())  # -> [2.0, -1.0, 0.5]

t3_nonzero = np.array([0.4, -0.2, 0.1])  # optional correction direction.
t3_scales = np.array([0.0, 0.5, 1.0])  # -> [0.0, 0.5, 1.0]
t3_outputs = np.array([t3_x + t3_s * t3_nonzero for t3_s in t3_scales])  # -> [[2.0, -1.0, 0.5], [2.2, -1.1, 0.55], [2.4, -1.2, 0.6]]

print("scaled residual outputs:", np.round(t3_outputs, 3).tolist())  # -> [[2.0, -1.0, 0.5], [2.2, -1.1, 0.55], [2.4, -1.2, 0.6]]

assert np.allclose(t3_identity, t3_x)
assert np.allclose(t3_outputs[0], t3_x)

plt.figure(figsize=(4.6, 2.8))
plt.plot(t3_scales, t3_outputs[:, 0], marker="o", label="coord0")
plt.plot(t3_scales, t3_outputs[:, 1], marker="s", label="coord1")
plt.plot(t3_scales, t3_outputs[:, 2], marker="^", label="coord2")
plt.title("Toy 3 · scale 0 is exact identity")
plt.xlabel("residual scale")
plt.ylabel("output")
plt.legend()
plt.show()

▶ What you'll see: at residual scale `0`, every output coordinate equals the input coordinate.

### ✍️ Toy 4 · Residual addition requires matching shapes

Elementwise addition can only pair coordinates when the shortcut and branch tensors have identical shapes.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)  # seeded for reproducibility.
t4_shortcut = np.ones((2, 2, 2))  # shortcut tensor.
t4_branch_good = 0.5 * np.ones((2, 2, 2))  # matching branch tensor.
t4_branch_bad = 0.5 * np.ones((2, 2, 3))  # mismatched channel count.

print("shortcut shape:", t4_shortcut.shape)  # -> (2, 2, 2)
print("good branch shape:", t4_branch_good.shape)  # -> (2, 2, 2)
print("bad branch shape:", t4_branch_bad.shape)  # -> (2, 2, 3)

t4_added = t4_shortcut + t4_branch_good  # -> first pixel [1.5, 1.5], shape (2, 2, 2)

print("valid sum shape:", t4_added.shape)  # -> (2, 2, 2)
print("valid sum first pixel:", t4_added[0, 0, :].tolist())  # -> [1.5, 1.5]

t4_can_add_bad = t4_shortcut.shape == t4_branch_bad.shape  # -> False

print("can add bad branch?", t4_can_add_bad)  # -> False

assert t4_added.shape == t4_shortcut.shape
assert t4_can_add_bad is False

plt.figure(figsize=(4.4, 2.8))
plt.bar(["shortcut C", "good C", "bad C"], [t4_shortcut.shape[2], t4_branch_good.shape[2], t4_branch_bad.shape[2]], color=["#4c78a8", "#54a24b", "#e45756"])
plt.title("Toy 4 · channel counts must match")
plt.ylabel("channels")
plt.show()

▶ What you'll see: the good branch adds cleanly, while the bad branch has 3 channels instead of 2.

### ✍️ Toy 5 · Projection shortcuts change channel coordinates

A `1×1` projection applies the same small matrix at each spatial position so the shortcut has the branch's channel count.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)  # seeded for reproducibility.
t5_x = np.array([[[1.0, 2.0], [0.0, 1.0]]])  # shape 1x2x2.
t5_W = np.array([[1.0, 0.0, 0.5], [0.0, 1.0, -1.0]])  # 2 input channels -> 3 output channels.

print("input shape:", t5_x.shape)  # -> (1, 2, 2)
print("projection matrix shape:", t5_W.shape)  # -> (2, 3)

t5_projected = t5_x @ t5_W  # -> [[[1.0, 2.0, -1.5], [0.0, 1.0, -1.0]]]

print("projected shortcut:", t5_projected.tolist())  # -> [[[1.0, 2.0, -1.5], [0.0, 1.0, -1.0]]]

t5_branch = 0.1 * np.ones_like(t5_projected)  # branch now matches projected shape.

print("branch shape:", t5_branch.shape)  # -> (1, 2, 3)

t5_y = t5_projected + t5_branch  # -> [[[1.1, 2.1, -1.4], [0.1, 1.1, -0.9]]]

print("residual output:", np.round(t5_y, 3).tolist())  # -> [[[1.1, 2.1, -1.4], [0.1, 1.1, -0.9]]]

t5_params = t5_W.size  # -> 6

print("projection weights:", t5_params)  # -> 6

assert t5_y.shape == (1, 2, 3)
assert t5_params == 6

plt.figure(figsize=(4.4, 2.8))
plt.imshow(t5_W, cmap="viridis", aspect="auto")
plt.colorbar(label="weight")
plt.title("Toy 5 · 1×1 projection matrix")
plt.xlabel("output channel")
plt.ylabel("input channel")
plt.show()

▶ What you'll see: the shortcut changes from 2 channels to 3 channels, making residual addition legal.

### ✍️ Toy 6 · The shortcut adds a direct gradient route

For `y = x + F(x)`, the backward multiplier includes a `1`, while a plain branch only has `dF/dx`.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)  # seeded for reproducibility.
t6_upstream = 3.0  # incoming dL/dy.
t6_dF_dx = 0.2  # learned branch derivative.

print("upstream gradient:", t6_upstream)  # -> 3.0
print("branch derivative dF/dx:", t6_dF_dx)  # -> 0.2

t6_res_grad = t6_upstream * (1.0 + t6_dF_dx)  # -> 3.5999999999999996

print("residual dL/dx:", round(float(t6_res_grad), 3))  # -> 3.6

t6_plain_grad = t6_upstream * t6_dF_dx  # -> 0.6000000000000001

print("plain branch-only dL/dx:", round(float(t6_plain_grad), 3))  # -> 0.6

t6_derivs = np.array([0.5, 0.4, 0.2])  # small branch derivatives.

print("deep branch derivatives:", t6_derivs.tolist())  # -> [0.5, 0.4, 0.2]

t6_plain_product = float(np.prod(t6_derivs))  # -> 0.04

print("plain product:", round(t6_plain_product, 3))  # -> 0.04

t6_res_product = float(np.prod(1.0 + t6_derivs))  # -> 2.5199999999999996

print("residual-style product:", round(t6_res_product, 3))  # -> 2.52

assert round(float(t6_res_grad), 3) == 3.6
assert round(t6_plain_product, 3) == 0.04

plt.figure(figsize=(4.6, 2.8))
plt.bar(["plain grad", "residual grad", "plain product", "identity product"], [t6_plain_grad, t6_res_grad, t6_plain_product, t6_res_product], color=["#e45756", "#54a24b", "#e45756", "#54a24b"])
plt.title("Toy 6 · identity term supports gradients")
plt.ylabel("multiplier / gradient")
plt.show()

▶ What you'll see: the residual gradient is `3.6` instead of `0.6`, and identity terms avoid the tiny product.

### ✍️ Toy 7 · Downsampling must align both residual paths

When spatial size and channels change, the shortcut must downsample and project to meet the residual branch before addition.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)  # seeded for reproducibility.
t7_x = np.arange(4 * 4 * 2, dtype=float).reshape(4, 4, 2)  # 4x4x2 input tensor.
t7_stride = 2  # -> 2
t7_Cout = 3  # -> 3

print("input shape:", t7_x.shape)  # -> (4, 4, 2)

t7_branch_shape = (t7_x.shape[0] // t7_stride, t7_x.shape[1] // t7_stride, t7_Cout)  # -> (2, 2, 3)

print("target branch shape:", t7_branch_shape)  # -> (2, 2, 3)

t7_raw_shortcut_shape = t7_x.shape  # -> (4, 4, 2)

print("raw shortcut shape:", t7_raw_shortcut_shape)  # -> (4, 4, 2)

t7_W = np.array([[1.0, 0.0, 0.5], [0.0, 1.0, -0.5]])  # 2 channels -> 3 channels.
t7_downsampled = t7_x[::t7_stride, ::t7_stride, :]  # -> shape (2, 2, 2)

print("downsampled shortcut shape:", t7_downsampled.shape)  # -> (2, 2, 2)

t7_projected = t7_downsampled @ t7_W  # -> shape (2, 2, 3)

print("projected shortcut shape:", t7_projected.shape)  # -> (2, 2, 3)

t7_branch = np.ones(t7_branch_shape)  # residual branch already has target shape.
t7_y = t7_projected + t7_branch  # -> shape (2, 2, 3)

print("added output shape:", t7_y.shape)  # -> (2, 2, 3)
print("first output vector:", t7_y[0, 0, :].tolist())  # -> [1.0, 2.0, 0.5]

t7_needs_projection = t7_raw_shortcut_shape != t7_branch_shape  # -> True

print("projection required?", t7_needs_projection)  # -> True

assert t7_y.shape == t7_branch_shape
assert t7_needs_projection is True

plt.figure(figsize=(4.8, 2.8))
plt.bar(["H before", "H after", "C before", "C after"], [4, 2, 2, 3], color=["#e45756", "#54a24b", "#e45756", "#54a24b"])
plt.title("Toy 7 · shortcut changes size and channels")
plt.ylabel("dimension")
plt.show()

▶ What you'll see: the raw shortcut `(4,4,2)` becomes `(2,2,3)` so it can add to the downsampled branch.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for vectors, small tensor arithmetic, and from-scratch residual blocks.
import matplotlib.pyplot as plt  # load Matplotlib for every diagnostic plot in the examples.
np.random.seed(0)  # make all random examples reproducible.

def relu(z):  # define the only nonlinearity used in the scratch residual branches.
    return np.maximum(0.0, z)  # apply ReLU elementwise.

def residual_add(x, f):  # define a safe residual addition helper.
    assert x.shape == f.shape  # require elementwise-compatible shortcut and residual branch.
    return x + f  # compute y = x + F(x).

def linear_residual(x, W, b):  # define a tiny residual branch F(x)=xW+b for vector demos.
    return x @ W + b  # return the learned correction.

def project_channels(x, W):  # define a 1x1-style projection on the last axis.
    return x @ W  # map C_in channels to C_out channels at every spatial position.

def conv2d_valid_single(image, kernel, stride=1):  # define a small valid 2-D convolution/correlation helper.
    out_h = (image.shape[0] - kernel.shape[0]) // stride + 1  # compute output height.
    out_w = (image.shape[1] - kernel.shape[1]) // stride + 1  # compute output width.
    out = np.zeros((out_h, out_w))  # allocate output feature map.
    for i in range(out_h):  # loop over output rows.
        for j in range(out_w):  # loop over output columns.
            patch = image[i * stride:i * stride + kernel.shape[0], j * stride:j * stride + kernel.shape[1]]  # select local patch.
            out[i, j] = np.sum(patch * kernel)  # compute one convolution response.
    return out  # return the feature map.

## 🟢 Basics (warm-up)

### Basic 1 — Add a residual correction

**Goal.** Compute $y=x+F(x)$ coordinate by coordinate, because residual learning starts with an elementwise correction. We build it in 2 steps.

In [ ]:
x_b1 = np.array([1.0, 2.0, 3.0])  # define the shortcut vector.
F_b1 = np.array([0.1, -0.2, 0.3])  # define the learned correction vector.

print("x:", x_b1)  # inspect the shortcut.
print("F(x):", F_b1)  # inspect the residual branch.

▶ What you'll see: the correction is small compared with the shortcut values.

In [ ]:
y_b1 = residual_add(x_b1, F_b1)  # add the shortcut and correction elementwise.

print("y = x + F(x):", y_b1)  # inspect the residual output.

assert np.allclose(y_b1, np.array([1.1, 1.8, 3.3]))  # verify the lesson arithmetic.
plt.figure(figsize=(4, 3))  # create a coordinate comparison plot.
plt.plot(x_b1, marker="o", label="x")  # plot shortcut values.
plt.plot(y_b1, marker="s", label="y")  # plot output values.
plt.title("Basic 1: residual correction")  # title the plot.
plt.legend()  # show labels.
plt.show()  # display the plot.

▶ What you'll see: the output line is the input line plus small coordinate shifts.

👀 Takeaway: a residual block changes the input only where the learned branch supplies a correction.

### Basic 2 — Make identity with zero correction

**Goal.** Set $F(x)=0$ and confirm the block returns its input, because easy identity is why extra residual depth is less dangerous. We build it in 2 steps.

In [ ]:
x_b2 = np.array([4.0, -1.0, 2.5])  # define a representation to preserve.
F_b2 = np.zeros_like(x_b2)  # define a zero residual branch.

print("zero correction:", F_b2)  # inspect F(x)=0.

▶ What you'll see: every correction coordinate is exactly zero.

In [ ]:
y_b2 = residual_add(x_b2, F_b2)  # compute y=x+0.

print("output:", y_b2)  # inspect output.

assert np.allclose(y_b2, x_b2)  # verify identity behavior.
plt.figure(figsize=(4, 3))  # create an identity plot.
plt.bar(["x0", "x1", "x2"], y_b2 - x_b2, color="seagreen")  # show output-input difference.
plt.axhline(0, color="black", linewidth=0.8)  # reference zero change.
plt.title("Basic 2: zero residual = identity")  # title the plot.
plt.show()  # display the plot.

▶ What you'll see: all bars sit at zero because the block made no change.

👀 Takeaway: residual blocks can choose “do nothing” by making the branch output zero.

### Basic 3 — Compare full target with correction

**Goal.** Measure how much smaller a correction can be than the full target, because residual learning simplifies near-identity mappings. We build it in 2 steps.

In [ ]:
x_b3 = np.array([1.0, 2.0, 3.0])  # define the input.
H_b3 = np.array([1.1, 1.8, 3.3])  # define the desired full output.
F_b3 = H_b3 - x_b3  # compute the residual target.

print("full target norm:", round(np.linalg.norm(H_b3), 3))  # inspect target size.
print("correction norm:", round(np.linalg.norm(F_b3), 3))  # inspect residual size.

▶ What you'll see: the correction norm is far smaller than the full target norm.

In [ ]:
ratio_b3 = np.linalg.norm(F_b3) / np.linalg.norm(H_b3)  # compute correction-to-target size ratio.

print("correction/target norm ratio:", round(ratio_b3, 3))  # inspect relative scale.

assert round(ratio_b3, 3) == 0.096  # verify the correction is about 9.6% of target size.
plt.figure(figsize=(4, 3))  # create a norm comparison chart.
plt.bar(["||H(x)||", "||F(x)||"], [np.linalg.norm(H_b3), np.linalg.norm(F_b3)], color=["gray", "teal"])  # compare magnitudes.
plt.title("Basic 3: correction is smaller")  # title the plot.
plt.show()  # display the chart.

▶ What you'll see: the residual target is a much shorter bar than the full output target.

👀 Takeaway: residual learning is easiest when the needed mapping is close to identity.

### Basic 4 — Reject mismatched shapes

**Goal.** Check shapes before adding, because residual addition is elementwise and cannot invent missing coordinates. We build it in 2 steps.

In [ ]:
x_b4 = np.ones((2, 3, 4))  # define a shortcut tensor with 4 channels.
F_b4 = np.ones((2, 3, 5))  # define a branch tensor with the wrong channel count.

print("x shape:", x_b4.shape)  # inspect shortcut shape.
print("F shape:", F_b4.shape)  # inspect residual branch shape.

▶ What you'll see: height and width match, but channels differ.

In [ ]:
same_shape_b4 = x_b4.shape == F_b4.shape  # explicitly test elementwise compatibility.

print("can add?", same_shape_b4)  # inspect legality.

assert same_shape_b4 is False  # verify that mismatched shapes are not addable.
plt.figure(figsize=(4, 3))  # create a small shape comparison plot.
plt.bar(["shortcut C", "branch C"], [x_b4.shape[-1], F_b4.shape[-1]], color=["teal", "crimson"])  # compare channels.
plt.title("Basic 4: channel mismatch")  # title the plot.
plt.show()  # display the chart.

▶ What you'll see: the channel bars differ, so a residual add would be invalid.

👀 Takeaway: shortcut and branch tensors must match in height, width, and channels before addition.

### Basic 5 — Add matching feature maps

**Goal.** Add two small feature maps with identical shape, because valid residual sums preserve tensor dimensions. We build it in 2 steps.

In [ ]:
x_b5 = np.arange(6, dtype=float).reshape(2, 3)  # define a 2x3 shortcut map.
F_b5 = 0.5 * np.ones_like(x_b5)  # define a matching correction map.

print("x shape:", x_b5.shape, "F shape:", F_b5.shape)  # inspect matching shapes.

▶ What you'll see: both arrays are 2×3.

In [ ]:
y_b5 = residual_add(x_b5, F_b5)  # compute the residual map.

print("y:\n", y_b5)  # inspect output values.

assert y_b5.shape == x_b5.shape  # verify addition preserves dimensions.
plt.figure(figsize=(4, 3))  # create a heatmap of the residual output.
plt.imshow(y_b5, cmap="viridis")  # visualize values.
plt.colorbar(label="value")  # add color scale.
plt.title("Basic 5: valid residual sum")  # title heatmap.
plt.show()  # display plot.

▶ What you'll see: every shortcut value is lifted by 0.5 with the same 2×3 shape.

👀 Takeaway: residual addition modifies values without changing tensor size when shapes already match.

### Basic 6 — Project channels with a 1×1 shortcut

**Goal.** Map 3 channels to 4 channels using a linear projection, because residual blocks sometimes need shortcuts that are not raw identity. We build it in 2 steps.

In [ ]:
x_b6 = np.array([[1.0, 2.0, 3.0], [0.0, 1.0, 0.5]])  # two spatial positions with three channels each.
W_b6 = np.array([[1.0, 0.0, 0.5, -0.5], [0.0, 1.0, 0.5, 0.5], [0.0, 0.0, 1.0, 1.0]])  # 3-to-4 channel projection.

print("x shape:", x_b6.shape, "W shape:", W_b6.shape)  # inspect matrix multiplication dimensions.

▶ What you'll see: the last dimension of x matches the first dimension of W.

In [ ]:
proj_b6 = project_channels(x_b6, W_b6)  # apply the 1x1-style projection.

print("projected shape:", proj_b6.shape)  # inspect output channel count.

assert proj_b6.shape == (2, 4)  # verify 3 channels became 4.
plt.figure(figsize=(4, 3))  # create a projection heatmap.
plt.imshow(proj_b6, cmap="viridis", aspect="auto")  # visualize projected features.
plt.colorbar(label="value")  # add color scale.
plt.title("Basic 6: projected shortcut")  # title heatmap.
plt.show()  # display plot.

▶ What you'll see: each spatial position now has four projected shortcut features.

👀 Takeaway: a projection shortcut changes channel count while keeping positions aligned.

### Basic 7 — Count projection parameters

**Goal.** Count weights in a $1\times1$ projection, because shape fixes have real model cost. We build it in 2 steps.

In [ ]:
cin_b7 = 64  # input channels.
cout_b7 = 128  # output channels.
k_b7 = 1  # 1x1 projection kernel size.
weights_b7 = k_b7 * k_b7 * cin_b7 * cout_b7  # count projection weights.

print("projection weights:", weights_b7)  # inspect parameter count.

assert weights_b7 == 8192  # verify the lesson's concrete number.

▶ What you'll see: the shortcut projection has 8192 learned weights.

In [ ]:
plt.figure(figsize=(4, 3))  # create a parameter-count plot.
plt.bar(["64→128 1×1"], [weights_b7], color="orange")  # plot the projection cost.
plt.title("Basic 7: projection is learned")  # title plot.
plt.ylabel("weights")  # label y-axis.
plt.show()  # display chart.

▶ What you'll see: even a 1×1 shortcut can add thousands of parameters.

👀 Takeaway: projection shortcuts are necessary sometimes, but they are learned layers, not free wires.

### Basic 8 — Compute a shortcut gradient

**Goal.** Evaluate $dL/dx=(dL/dy)(1+dF/dx)$, because residual blocks add an identity derivative to the backward path. We build it in 2 steps.

In [ ]:
dL_dy_b8 = 2.0  # upstream scalar gradient.
dF_dx_b8 = 0.1  # derivative through the residual branch.
grad_b8 = dL_dy_b8 * (1 + dF_dx_b8)  # residual gradient to x.
branch_only_b8 = dL_dy_b8 * dF_dx_b8  # gradient without identity shortcut.

print("residual gradient:", round(grad_b8, 3))  # inspect shortcut-aided gradient.
print("branch-only gradient:", round(branch_only_b8, 3))  # inspect branch-only contribution.

assert round(grad_b8, 3) == 2.2  # verify the lesson value.

▶ What you'll see: the identity term makes the gradient much larger than the branch-only path.

In [ ]:
plt.figure(figsize=(4, 3))  # create gradient comparison plot.
plt.bar(["branch only", "residual"], [branch_only_b8, grad_b8], color=["crimson", "seagreen"])  # compare gradients.
plt.title("Basic 8: shortcut derivative")  # title plot.
plt.ylabel("dL/dx")  # label gradient scale.
plt.show()  # display chart.

▶ What you'll see: the residual bar includes the direct identity route.

👀 Takeaway: residual shortcuts improve gradient flow by adding a direct derivative term.

### Basic 9 — Downsample both paths

**Goal.** Compute the output shape for a stride-2 residual block, because both shortcut and branch must land on the same grid. We build it in 2 steps.

In [ ]:
in_shape_b9 = (56, 56, 64)  # incoming height, width, channels.
stride_b9 = 2  # downsampling stride.
out_channels_b9 = 128  # widened output channels.
out_shape_b9 = (in_shape_b9[0] // stride_b9, in_shape_b9[1] // stride_b9, out_channels_b9)  # target block shape.

print("input shape:", in_shape_b9)  # inspect input.
print("output shape:", out_shape_b9)  # inspect downsampled target.

assert out_shape_b9 == (28, 28, 128)  # verify shape arithmetic.

▶ What you'll see: stride 2 halves 56×56 to 28×28 and the block widens to 128 channels.

In [ ]:
plt.figure(figsize=(4, 3))  # create shape-change chart.
plt.bar(["H", "W", "C"], in_shape_b9, alpha=0.6, label="input")  # plot input dimensions.
plt.bar(["H", "W", "C"], out_shape_b9, alpha=0.6, label="output")  # plot output dimensions.
plt.title("Basic 9: residual downsampling")  # title chart.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: spatial dimensions shrink while channel count grows.

👀 Takeaway: downsampling residual blocks need a strided projection shortcut to match the branch.

### Basic 10 — Residual block as a tiny function

**Goal.** Build a small vector residual block $y=x+\operatorname{ReLU}(xW+b)$, because ResNet blocks are ordinary functions wrapped by a shortcut. We build it in 3 steps.

In [ ]:
x_b10 = np.array([1.0, -2.0, 0.5])  # define one input vector.
W_b10 = np.array([[0.1, 0.0, 0.2], [0.0, -0.1, 0.1], [0.2, 0.1, 0.0]])  # residual branch weights.
b_b10 = np.array([0.0, 0.1, -0.1])  # residual branch bias.

print("x:", x_b10)  # inspect input.

▶ What you'll see: the vector has positive and negative coordinates.

In [ ]:
raw_b10 = linear_residual(x_b10, W_b10, b_b10)  # compute pre-activation correction.
F_b10 = relu(raw_b10)  # apply ReLU to make a nonlinear residual branch.

print("raw branch:", np.round(raw_b10, 3))  # inspect branch before nonlinearity.
print("F(x):", np.round(F_b10, 3))  # inspect nonnegative correction after ReLU.

In [ ]:
y_b10 = residual_add(x_b10, F_b10)  # add the shortcut.

print("y:", np.round(y_b10, 3))  # inspect residual block output.

assert y_b10.shape == x_b10.shape  # verify the block preserves vector shape.
plt.figure(figsize=(4, 3))  # create before/after plot.
plt.plot(x_b10, marker="o", label="x")  # plot input.
plt.plot(y_b10, marker="s", label="y")  # plot output.
plt.title("Basic 10: tiny residual block")  # title plot.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: the nonlinear branch changes only coordinates where ReLU leaves a positive correction.

👀 Takeaway: a residual block is a learned transformation plus an explicit shortcut of matching shape.

## 🟡 Easy

### Easy 1 — Compare plain and residual targets

**Goal.** Train a one-layer linear map to fit a near-identity target two ways, because the residual version learns smaller numbers. We build it in 3 steps.

In [ ]:
X_e1 = np.eye(3)  # three basis inputs make the target mapping easy to inspect.
H_e1 = np.array([[1.1, 0.0, 0.0], [0.0, 0.8, 0.0], [0.0, 0.0, 1.3]])  # full near-identity target.
F_target_e1 = H_e1 - X_e1  # residual target equals correction matrix.

print("full target:\n", H_e1)  # inspect H.
print("residual target:\n", F_target_e1)  # inspect H-I.

▶ What you'll see: the residual target has zeros off-diagonal and small diagonal corrections.

In [ ]:
plain_size_e1 = np.linalg.norm(H_e1)  # measure full target size.
res_size_e1 = np.linalg.norm(F_target_e1)  # measure correction size.

print("||H||:", round(plain_size_e1, 3), "||H-I||:", round(res_size_e1, 3))  # compare sizes.

assert round(res_size_e1, 3) == 0.374  # verify correction magnitude.

In [ ]:
plt.figure(figsize=(4, 3))  # create size comparison chart.
plt.bar(["plain H", "residual H-I"], [plain_size_e1, res_size_e1], color=["gray", "teal"])  # compare norms.
plt.title("Easy 1: residual target is smaller")  # title chart.
plt.ylabel("Frobenius norm")  # label norm scale.
plt.show()  # display plot.

▶ What you'll see: the correction target is far smaller than the full mapping target.

👀 Takeaway: residual learning can turn a near-identity task into a small-correction task.

### Easy 2 — Build a projection residual block

**Goal.** Use a projection shortcut and a matching residual branch, because channel-changing blocks cannot add raw identity. We build it in 3 steps.

In [ ]:
x_e2 = np.array([[1.0, 2.0], [3.0, 4.0]])  # two positions with two channels.
W_short_e2 = np.array([[1.0, 0.0, 0.5], [0.0, 1.0, -0.5]])  # shortcut projection 2->3.
W_branch_e2 = np.array([[0.1, 0.2, 0.0], [0.0, -0.1, 0.3]])  # residual branch also outputs 3 channels.

print("x shape:", x_e2.shape)  # inspect input shape.

▶ What you'll see: the input has two channels, but the block will output three.

In [ ]:
shortcut_e2 = project_channels(x_e2, W_short_e2)  # project shortcut to 3 channels.
F_e2 = project_channels(x_e2, W_branch_e2)  # compute branch correction in same 3-channel space.
y_e2 = residual_add(shortcut_e2, F_e2)  # add after shapes match.

print("shortcut shape:", shortcut_e2.shape, "branch shape:", F_e2.shape)  # inspect aligned shapes.
print("output:\n", np.round(y_e2, 3))  # inspect projected residual output.

assert y_e2.shape == (2, 3)  # verify output channel count.

In [ ]:
plt.figure(figsize=(4, 3))  # create output heatmap.
plt.imshow(y_e2, cmap="viridis", aspect="auto")  # visualize output features.
plt.colorbar(label="value")  # add color scale.
plt.title("Easy 2: projected residual block")  # title heatmap.
plt.show()  # display plot.

▶ What you'll see: both paths are 3-channel tensors before addition, so the output is valid.

👀 Takeaway: projection aligns the shortcut with the residual branch's output coordinate system.

### Easy 3 — Simulate gradient flow through many blocks

**Goal.** Multiply simple gradient factors through depth, because residual identity terms reduce vanishing compared with plain branch products. We build it in 3 steps.

In [ ]:
derivs_e3 = np.array([0.9, 0.7, 0.5, 0.3, 0.2])  # local branch derivatives across five blocks.
plain_running_e3 = np.cumprod(derivs_e3)  # plain stack gradient multipliers.
res_running_e3 = np.cumprod(1.0 + derivs_e3)  # residual-style multipliers with identity terms.

print("plain cumulative:", np.round(plain_running_e3, 4))  # inspect shrinking product.
print("residual cumulative:", np.round(res_running_e3, 4))  # inspect identity-supported product.

▶ What you'll see: the plain product quickly shrinks as depth increases.

In [ ]:
assert round(float(plain_running_e3[-1]), 4) == 0.0189  # verify final plain multiplier.

print("final ratio residual/plain:", round(float(res_running_e3[-1] / plain_running_e3[-1]), 1))  # inspect the gap.

In [ ]:
plt.figure(figsize=(5, 3))  # create depth curve.
plt.plot(plain_running_e3, marker="o", label="plain product")  # plot plain multipliers.
plt.plot(res_running_e3, marker="s", label="with identity terms")  # plot residual-style multipliers.
plt.yscale("log")  # use log scale so both curves fit.
plt.title("Easy 3: gradient multipliers through depth")  # title plot.
plt.xlabel("block index")  # label x-axis.
plt.ylabel("multiplier, log scale")  # label y-axis.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: on a log scale, the plain multiplier falls while identity terms keep a stronger path.

👀 Takeaway: residual blocks help gradients cross depth by adding identity derivatives at every block.

### Easy 4 — Compare addition and concatenation

**Goal.** Contrast residual addition with feature concatenation, because addition keeps width fixed while concatenation grows channels. We build it in 3 steps.

In [ ]:
x_e4 = np.ones((2, 2, 3))  # shortcut feature map with 3 channels.
F_e4 = 2 * np.ones((2, 2, 3))  # matching residual branch with 3 channels.
add_e4 = x_e4 + F_e4  # ResNet-style addition.
cat_e4 = np.concatenate([x_e4, F_e4], axis=-1)  # DenseNet-style concatenation.

print("add shape:", add_e4.shape)  # inspect fixed channel count.
print("concat shape:", cat_e4.shape)  # inspect doubled channel count.

▶ What you'll see: addition returns 3 channels, concatenation returns 6.

In [ ]:
assert add_e4.shape[-1] == 3  # verify addition keeps channel count.
assert cat_e4.shape[-1] == 6  # verify concatenation grows channel count.
channel_counts_e4 = [x_e4.shape[-1], add_e4.shape[-1], cat_e4.shape[-1]]  # collect widths.

print("channel counts:", channel_counts_e4)  # inspect widths.

In [ ]:
plt.figure(figsize=(4, 3))  # create channel-count chart.
plt.bar(["input", "add", "concat"], channel_counts_e4, color=["gray", "teal", "orange"])  # compare widths.
plt.title("Easy 4: add vs concatenate")  # title chart.
plt.ylabel("channels")  # label y-axis.
plt.show()  # display plot.

▶ What you'll see: concatenation widens the interface; residual addition keeps it fixed.

👀 Takeaway: ResNet addition is efficient but requires matching coordinate meanings.

### Easy 5 — Build a two-block residual stack

**Goal.** Stack two small residual blocks, because ResNets compose many small corrections instead of one large rewrite. We build it in 3 steps.

In [ ]:
x_e5 = np.array([1.0, 0.5])  # input vector.
W1_e5 = np.array([[0.1, 0.0], [0.2, -0.1]])  # first branch weights.
W2_e5 = np.array([[0.0, 0.2], [-0.1, 0.1]])  # second branch weights.

print("start:", x_e5)  # inspect starting vector.

▶ What you'll see: the stack begins from a simple two-coordinate representation.

In [ ]:
F1_e5 = relu(x_e5 @ W1_e5)  # first block correction.
h1_e5 = residual_add(x_e5, F1_e5)  # first block output.
F2_e5 = relu(h1_e5 @ W2_e5)  # second block correction.
y_e5 = residual_add(h1_e5, F2_e5)  # second block output.

print("after block 1:", np.round(h1_e5, 3))  # inspect intermediate representation.
print("after block 2:", np.round(y_e5, 3))  # inspect final representation.

assert y_e5.shape == x_e5.shape  # verify shape is preserved across blocks.

In [ ]:
plt.figure(figsize=(4, 3))  # create representation trajectory plot.
plt.plot([x_e5[0], h1_e5[0], y_e5[0]], marker="o", label="coord0")  # coordinate 0 over blocks.
plt.plot([x_e5[1], h1_e5[1], y_e5[1]], marker="s", label="coord1")  # coordinate 1 over blocks.
plt.xticks([0, 1, 2], ["input", "block1", "block2"])  # label block positions.
plt.title("Easy 5: stacked residual corrections")  # title plot.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: each block nudges the representation instead of replacing it from scratch.

👀 Takeaway: deep residual networks are long compositions of small, shape-preserving corrections.

## 🔴 Advanced

### Advanced 1 — Train residual versus plain on a near-identity map

**Goal.** Optimize a linear plain map and a linear residual correction on the same near-identity data, because residual parameterization starts closer to a good solution. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(1)  # reproducible synthetic data.
X_a1 = rng_a1.normal(size=(80, 3))  # training inputs.
A_true_a1 = np.eye(3) + np.diag([0.1, -0.2, 0.3])  # near-identity target map.
Y_a1 = X_a1 @ A_true_a1  # target outputs.

print("data shape:", X_a1.shape)  # inspect training set.

▶ What you'll see: eighty 3-D examples define a near-identity supervised task.

In [ ]:
W_plain_a1 = np.zeros((3, 3))  # plain model starts from zero output.
W_res_a1 = np.zeros((3, 3))  # residual model starts from identity plus zero correction.
eta_a1 = 0.2  # learning rate for full-batch gradient descent.
loss_plain_a1 = []  # store plain losses.
loss_res_a1 = []  # store residual losses.

print("initial plain prediction mean:", round(float(np.mean(X_a1 @ W_plain_a1)), 3))  # inspect plain start.

In [ ]:
for step_a1 in range(40):  # run a small deterministic training loop.
    P_plain_a1 = X_a1 @ W_plain_a1  # plain predictions.
    P_res_a1 = X_a1 + X_a1 @ W_res_a1  # residual predictions.
    E_plain_a1 = P_plain_a1 - Y_a1  # plain error.
    E_res_a1 = P_res_a1 - Y_a1  # residual error.
    loss_plain_a1.append(float(np.mean(E_plain_a1 ** 2)))  # record plain MSE.
    loss_res_a1.append(float(np.mean(E_res_a1 ** 2)))  # record residual MSE.
    W_plain_a1 -= eta_a1 * (X_a1.T @ E_plain_a1) / X_a1.shape[0]  # gradient step for plain W.
    W_res_a1 -= eta_a1 * (X_a1.T @ E_res_a1) / X_a1.shape[0]  # gradient step for residual correction W.

print("initial losses:", round(loss_plain_a1[0], 3), round(loss_res_a1[0], 3))  # compare starts.
print("final losses:", round(loss_plain_a1[-1], 5), round(loss_res_a1[-1], 5))  # compare endings.

assert loss_res_a1[0] < loss_plain_a1[0]  # residual starts closer because identity is built in.

In [ ]:
plt.figure(figsize=(5, 3))  # create training curve plot.
plt.plot(loss_plain_a1, label="plain learns H")  # plot plain loss.
plt.plot(loss_res_a1, label="residual learns H-I")  # plot residual loss.
plt.yscale("log")  # show early advantage and convergence.
plt.title("Advanced 1: near-identity optimization")  # title plot.
plt.xlabel("step")  # label x-axis.
plt.ylabel("MSE, log scale")  # label y-axis.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: the residual model begins with much lower error because it already contains the identity map.

👀 Takeaway: residual parameterization changes the optimization problem even when model capacity is similar.

### Advanced 2 — Validate projection for downsampling

**Goal.** Simulate a stride-2, channel-changing shortcut and branch, because downsampling residual blocks must align both paths before addition. We build it in 4 steps.

In [ ]:
x_a2 = np.arange(4 * 4 * 2, dtype=float).reshape(4, 4, 2)  # 4x4 map with 2 channels.
W_short_a2 = np.array([[1.0, 0.5, 0.0], [0.0, 0.5, 1.0]])  # shortcut projection 2->3.
W_branch_a2 = np.array([[0.1, 0.0, 0.2], [0.0, -0.1, 0.3]])  # branch projection 2->3 after stride.

print("input shape:", x_a2.shape)  # inspect input.

▶ What you'll see: the toy input has a small spatial grid and two channels.

In [ ]:
x_stride_a2 = x_a2[::2, ::2, :]  # stride-2 spatial subsampling for both paths.
shortcut_a2 = project_channels(x_stride_a2, W_short_a2)  # projected shortcut.
branch_a2 = project_channels(x_stride_a2, W_branch_a2)  # matching residual branch.

print("strided shape:", x_stride_a2.shape)  # inspect 2x2 spatial size.
print("shortcut shape:", shortcut_a2.shape, "branch shape:", branch_a2.shape)  # inspect aligned outputs.

assert shortcut_a2.shape == branch_a2.shape == (2, 2, 3)  # verify downsampled channel-aligned shape.

In [ ]:
y_a2 = residual_add(shortcut_a2, branch_a2)  # add after downsampling and projection.

print("output shape:", y_a2.shape)  # inspect final block shape.

assert y_a2.shape == (2, 2, 3)  # verify output shape.

In [ ]:
plt.figure(figsize=(4, 3))  # create heatmap for one output channel.
plt.imshow(y_a2[:, :, 0], cmap="viridis")  # visualize first output channel.
plt.colorbar(label="channel 0 value")  # add color scale.
plt.title("Advanced 2: downsampled residual output")  # title plot.
plt.show()  # display heatmap.

▶ What you'll see: the output grid is 2×2 with projected channels, so the residual sum is legal.

👀 Takeaway: stride and projection must be mirrored on the shortcut when the residual branch changes resolution or width.

### Advanced 3 — Sweep residual scale in a deep stack

**Goal.** Vary the scale of residual corrections through many blocks, because small residual branches preserve stability while still allowing gradual change. We build it in 4 steps.

In [ ]:
x0_a3 = np.array([1.0, 0.5])  # initial representation.
W_a3 = np.array([[0.2, -0.1], [0.1, 0.15]])  # shared residual branch weights.
scales_a3 = np.array([0.0, 0.25, 0.5, 1.0])  # residual scaling values to compare.

print("scales:", scales_a3)  # inspect sweep.

▶ What you'll see: scale 0 is pure identity and scale 1 applies the full branch.

In [ ]:
trajectories_a3 = []  # store final vectors for each scale.
for scale_a3 in scales_a3:  # loop over residual branch scales.
    h_a3 = x0_a3.copy()  # reset representation.
    for block_a3 in range(12):  # compose many residual blocks.
        h_a3 = h_a3 + scale_a3 * np.tanh(h_a3 @ W_a3)  # residual update with bounded correction.
    trajectories_a3.append(h_a3)  # save final representation.
trajectories_a3 = np.array(trajectories_a3)  # convert to array for plotting.

print("final vectors:\n", np.round(trajectories_a3, 3))  # inspect final states.

In [ ]:
drift_a3 = np.linalg.norm(trajectories_a3 - x0_a3, axis=1)  # measure how far each scale moves from identity.

print("drift from input:", np.round(drift_a3, 3))  # inspect stability/change tradeoff.

assert round(float(drift_a3[0]), 3) == 0.0  # scale 0 remains exact identity.

In [ ]:
plt.figure(figsize=(5, 3))  # create drift plot.
plt.plot(scales_a3, drift_a3, marker="o", color="purple")  # plot drift by residual scale.
plt.title("Advanced 3: residual scale controls drift")  # title plot.
plt.xlabel("residual branch scale")  # label x-axis.
plt.ylabel("||final - input||")  # label drift.
plt.show()  # display plot.

▶ What you'll see: larger residual scales move the representation farther from the identity path.

👀 Takeaway: residual blocks make depth stable because each block can be a small controlled update.

### Advanced 4 — Detect incompatible semantic addition

**Goal.** Show that matching shape is necessary but not sufficient, because addition assumes channels have compatible meanings. We build it in 4 steps.

In [ ]:
edges_a4 = np.array([2.0, 0.5, 1.0])  # imagine channels representing edge-like features.
colors_a4 = np.array([0.1, 3.0, 2.0])  # imagine same shape but color-like features.

print("edges shape:", edges_a4.shape, "colors shape:", colors_a4.shape)  # inspect shape compatibility.

▶ What you'll see: the two vectors can be added numerically because their shapes match.

In [ ]:
bad_sum_a4 = edges_a4 + colors_a4  # shape-valid but semantically questionable addition.
projection_a4 = np.array([[0.1, 0.0, 0.0], [0.0, 0.2, 0.0], [0.0, 0.0, 0.3]])  # learned alignment example.
aligned_colors_a4 = colors_a4 @ projection_a4  # map color-like coordinates into edge-like scale.
good_sum_a4 = edges_a4 + aligned_colors_a4  # add after a simple learned alignment.

print("shape-valid raw sum:", bad_sum_a4)  # inspect careless sum.
print("aligned sum:", np.round(good_sum_a4, 3))  # inspect projected sum.

In [ ]:
raw_change_a4 = np.linalg.norm(bad_sum_a4 - edges_a4)  # measure unaligned perturbation size.
aligned_change_a4 = np.linalg.norm(good_sum_a4 - edges_a4)  # measure aligned perturbation size.

print("raw change norm:", round(raw_change_a4, 3), "aligned change norm:", round(aligned_change_a4, 3))  # compare changes.

assert aligned_change_a4 < raw_change_a4  # projection controls coordinate mixing in this demo.

In [ ]:
plt.figure(figsize=(4, 3))  # create semantic-change comparison plot.
plt.bar(["raw add", "projected add"], [raw_change_a4, aligned_change_a4], color=["crimson", "seagreen"])  # compare perturbation sizes.
plt.title("Advanced 4: shape ≠ semantics")  # title plot.
plt.ylabel("change norm")  # label y-axis.
plt.show()  # display plot.

▶ What you'll see: both additions are legal by shape, but projection controls how much incompatible coordinates disturb the shortcut.

👀 Takeaway: residual addition assumes the two paths are in a compatible feature coordinate system, not merely the same array shape.

### Advanced 5 — Build a tiny ResNet classifier from scratch

**Goal.** Train a small NumPy residual network on a synthetic two-class problem, because residual blocks are useful only if gradients can optimize real weights. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(5)  # reproducible synthetic classification data.
X_a5 = rng_a5.normal(size=(120, 2))  # two-dimensional inputs.
y_a5 = (X_a5[:, 0] + X_a5[:, 1] > 0).astype(float)  # linear target labels.

print("class balance:", round(float(y_a5.mean()), 3))  # inspect positive fraction.

▶ What you'll see: the labels are reasonably balanced around a diagonal boundary.

In [ ]:
W1_a5 = 0.1 * rng_a5.normal(size=(2, 2))  # residual branch weights.
Wout_a5 = 0.1 * rng_a5.normal(size=2)  # output classifier weights.
bout_a5 = 0.0  # output bias.
losses_a5 = []  # store logistic losses.

print("initial W1 norm:", round(float(np.linalg.norm(W1_a5)), 3))  # inspect small initialization.

In [ ]:
for step_a5 in range(300):  # train by full-batch gradient descent.
    Z_a5 = X_a5 @ W1_a5  # residual branch pre-activation.
    F_a5 = np.tanh(Z_a5)  # bounded residual correction.
    H_a5 = X_a5 + F_a5  # residual hidden representation.
    logits_a5 = H_a5 @ Wout_a5 + bout_a5  # classifier scores.
    probs_a5 = 1.0 / (1.0 + np.exp(-logits_a5))  # sigmoid probabilities.
    loss_a5 = -np.mean(y_a5 * np.log(probs_a5 + 1e-9) + (1 - y_a5) * np.log(1 - probs_a5 + 1e-9))  # logistic loss.
    losses_a5.append(float(loss_a5))  # record loss.
    dlogits_a5 = (probs_a5 - y_a5) / X_a5.shape[0]  # derivative of mean logistic loss.
    dWout_a5 = H_a5.T @ dlogits_a5  # output weight gradient.
    dbout_a5 = float(np.sum(dlogits_a5))  # output bias gradient.
    dH_a5 = dlogits_a5[:, None] * Wout_a5[None, :]  # gradient into residual hidden state.
    dF_a5 = dH_a5  # gradient through addition to residual branch.
    dZ_a5 = dF_a5 * (1 - np.tanh(Z_a5) ** 2)  # tanh derivative.
    dW1_a5 = X_a5.T @ dZ_a5  # branch weight gradient.
    Wout_a5 -= 0.5 * dWout_a5  # update classifier weights.
    bout_a5 -= 0.5 * dbout_a5  # update classifier bias.
    W1_a5 -= 0.5 * dW1_a5  # update residual branch weights.

print("loss start -> end:", round(losses_a5[0], 3), "->", round(losses_a5[-1], 3))  # inspect training progress.

assert losses_a5[-1] < losses_a5[0]  # verify optimization improved the model.

In [ ]:
Z_a5 = X_a5 @ W1_a5  # recompute trained branch.
H_a5 = X_a5 + np.tanh(Z_a5)  # trained residual hidden representation.
probs_a5 = 1.0 / (1.0 + np.exp(-(H_a5 @ Wout_a5 + bout_a5)))  # trained probabilities.
acc_a5 = np.mean((probs_a5 >= 0.5) == y_a5)  # compute training accuracy.

print("training accuracy:", round(float(acc_a5), 3))  # inspect fit quality.

assert acc_a5 > 0.9  # verify the tiny residual classifier learned the boundary.

In [ ]:
plt.figure(figsize=(5, 3))  # create loss curve plot.
plt.plot(losses_a5, color="teal")  # plot logistic loss.
plt.title("Advanced 5: tiny residual classifier trains")  # title plot.
plt.xlabel("step")  # label training step.
plt.ylabel("logistic loss")  # label loss.
plt.show()  # display curve.

▶ What you'll see: the loss decreases and accuracy exceeds 90%, showing that the residual branch can be optimized from scratch.

👀 Takeaway: residual learning is an architectural parameterization that supports real gradient-based training, not just a forward-pass trick.